In [ ]:
!pip -q install ultralytics scikit-learn pyyaml pandas

In [ ]:
#import libraries
import csv
import cv2
from ultralytics import YOLO
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import Video, display
from pathlib import Path

In [ ]:
# Select Platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
#Project Path
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/Final_Thesis")

else:
    PROJECT_ROOT = Path.cwd()
print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

In [ ]:
# PATH
DATASET_ROOT = PROJECT_ROOT / "dataset"
data = DATASET_ROOT/"data.yaml"
RUNS_PATH = PROJECT_ROOT / "outputs"/"runs"
vdo_path = PROJECT_ROOT /"vdo"/"15sec.mp4"
tracker_path = PROJECT_ROOT /"tracker"

#Create tracker path
tracker_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Paths
vdo_path = Path(vdo_path)
runs_path = Path(RUNS_PATH)
tracker_path = Path(tracker_path)


# Model path
model_path = runs_path / "yolov11s" / "weights" / "best.pt"
model = YOLO(str(model_path))

# Open video
cap = cv2.VideoCapture(str(vdo_path))
if not cap.isOpened():
    raise ValueError(f"Could not open video: {vdo_path}")

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 29.97  # fallback

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_idx = 0
unique_track_ids = set()

# Output folder
track_runs_path = runs_path / "track_matrices"
track_runs_path.mkdir(parents=True, exist_ok=True)

# Output video path
output_video_path = track_runs_path / "tracked_matrices_bytetrack_iou0.7conf0.7.mp4"

# Video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

# CSV output
csv_path = track_runs_path / "bbox_centers_xyxy_bytetrack_iou0.7conf0.7.csv"
csv_file = open(str(csv_path), mode="w", newline="", encoding="utf-8")
writer = csv.writer(csv_file)
writer.writerow([
    "frame_idx", "time_sec", "track_id",
    "x1", "y1", "x2", "y2", "x_center", "y_center"
])

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    # Track objects
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.7,
        iou=0.7,
        verbose=False
    )

    annotated_frame = results[0].plot()

    if results[0].boxes is not None and results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.cpu().numpy()
        boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()
        time_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        for track_id, box in zip(track_ids, boxes_xyxy):
            x1, y1, x2, y2 = box
            x_center = (x1 + x2) / 2
            y_center = (y1 + y2) / 2

            writer.writerow([
                frame_idx, time_sec, int(track_id),
                float(x1), float(y1), float(x2), float(y2),
                float(x_center), float(y_center)
            ])

            unique_track_ids.add(int(track_id))

    cv2.putText(
        annotated_frame,
        f"Track IDs: {len(unique_track_ids)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    out.write(annotated_frame)

    # Optional preview every 100 frames
    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}")
        # Uncomment to preview frames in Colab:
        # cv2_imshow(annotated_frame)

# Cleanup
cap.release()
out.release()
csv_file.close()

print("Processing complete.")
print(f"Output video saved to: {output_video_path}")
print(f"CSV saved to: {csv_path}")

# Display the saved video in Colab
display(Video(str(output_video_path), embed=True))

In [ ]:
import pandas as pd

# paths
csv_path = RUNS_PATH/"track_matrices/bbox_centers_xyxy_bytetrack_iou0.7conf0.7.csv"
mot_pred_path = RUNS_PATH/"track_matrices/pred_mot_bytetrack_iou0.7conf0.7.txt"

# read your tracker csv
df = pd.read_csv(csv_path)

# convert xyxy -> xywh
df["bb_left"] = df["x1"]
df["bb_top"] = df["y1"]
df["bb_width"] = df["x2"] - df["x1"]
df["bb_height"] = df["y2"] - df["y1"]

# build MOT dataframe
mot_df = pd.DataFrame({
    "frame": df["frame_idx"].astype(int),
    "id": df["track_id"].astype(int),
    "bb_left": df["bb_left"].astype(float),
    "bb_top": df["bb_top"].astype(float),
    "bb_width": df["bb_width"].astype(float),
    "bb_height": df["bb_height"].astype(float),
    "conf": 1.0,   # use 1.0 if confidence is not saved
    "x": -1,
    "y": -1,
    "z": -1
})

# save as txt, no header
mot_df.to_csv(mot_pred_path, header=False, index=False)
print("Saved MOT prediction file to:", mot_pred_path)
print(mot_df.head())

Saved MOT prediction file to: /content/drive/MyDrive/Final_Thesis/runs/track_matrices/pred_mot_bytetrack_iou0.7conf0.7.txt
   frame  id      bb_left      bb_top    bb_width  bb_height  conf  x  y  z
0      1   1  1051.354736  404.667480  170.963135  38.223450   1.0 -1 -1 -1
1      1   2   571.593323  441.331970  171.505127  46.716980   1.0 -1 -1 -1
2      1   3   795.619629  465.257904  170.469360  36.973938   1.0 -1 -1 -1
3      1   4  1146.312744  318.096802  172.583252  39.275269   1.0 -1 -1 -1
4      1   5   571.907349  584.223633  172.461975  34.885071   1.0 -1 -1 -1


In [ ]:
import pandas as pd

# Path to the GT file exported from CVAT
gt_cvat_path = RUNS_PATH/"track_matrices/gt_15sec.txt"

# Output MOT-compatible GT file
gt_mot_path = RUNS_PATH/"track_matrices/gt_mot.txt"

# Read CVAT export
gt = pd.read_csv(gt_cvat_path, header=None)

# Keep only first 6 columns:
# frame, id, x, y, w, h
gt_mot = gt.iloc[:, :6].copy()

# Add MOT required columns
gt_mot[6] = 1    # confidence
gt_mot[7] = -1
gt_mot[8] = -1
gt_mot[9] = -1

# Save as MOT txt
gt_mot.to_csv(gt_mot_path, header=False, index=False)

print("Saved MOT-compatible GT file:", gt_mot_path)
print(gt_mot.head())

Saved MOT-compatible GT file: /content/drive/MyDrive/thesis-vti/runs_balsta/track_matrices/gt_mot.txt
   0  1        2       3       4      5  6  7  8  9
0  1  1   867.53  285.78  217.53  34.84  1 -1 -1 -1
1  1  2  1143.80  316.78  174.65  40.65  1 -1 -1 -1
2  1  3  1051.30  403.40  172.10  40.20  1 -1 -1 -1
3  1  4   797.09  465.30  170.21  35.78  1 -1 -1 -1
4  1  5   569.52  442.00  175.68  47.56  1 -1 -1 -1


In [ ]:
!pip uninstall -y numpy motmetrics
!pip install numpy==1.26.4 motmetrics

Found existing installation: numpy 2.4.4
Uninstalling numpy-2.4.4:
  Successfully uninstalled numpy-2.4.4
Found existing installation: motmetrics 1.4.0
Uninstalling motmetrics-1.4.0:
  Successfully uninstalled motmetrics-1.4.0
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached motmetrics-1.4.0-py3-none-any.whl.metadata (20 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached motmetrics-1.4.0-py3-none-any.whl (161 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 

In [ ]:
import numpy as np

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

import motmetrics as mm

gt_file = RUNS_PATH/"track_matrices/gt_mot.txt"
pred_file = RUNS_PATH/"track_matrices/pred_mot_bytetrack_iou0.7conf0.7.txt"

gt = mm.io.loadtxt(gt_file, fmt="mot15-2D", min_confidence=1)
pred = mm.io.loadtxt(pred_file, fmt="mot15-2D")

acc = mm.utils.compare_to_groundtruth(gt, pred, dist="iou", distth=0.5)

mh = mm.metrics.create()
summary = mh.compute(
    acc,
    metrics=[
        "num_frames",
        "mota",
        "motp",
        "idf1",
        "precision",
        "recall",
        "num_switches",
        "num_false_positives",
        "num_misses",
    ],
    name="bytetrack_iou0.7conf0.7"
)

print(summary)

                         num_frames  mota      motp  idf1  precision  recall  \
bytetrack_iou0.7conf0.7         451   1.0  0.128387   1.0        1.0     1.0   

                         num_switches  num_false_positives  num_misses  
bytetrack_iou0.7conf0.7             0                    0           0  
